# 🖥️ Delayed starts


In many applications, it is needed to 'delay' the start of particle advection. For example because particles need to be released at different times throughout an experiment. Or because particles need to be released at a constant rate from the same set of locations.

This tutorial will show how this can be done. We start with importing the relevant modules.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import xarray as xr
from matplotlib.animation import FuncAnimation

import parcels
import parcels.tutorial

# for interactive display of animations
plt.rcParams["animation.html"] = "jshtml"

First import a `FieldSet` (from the Argo example, in this case)


In [ ]:
# Load the CopernicusMarine data in the Agulhas region from the example_datasets
ds_fields = parcels.tutorial.open_dataset(
    "CopernicusMarine_data_for_Argo_tutorial/data"
)
ds_fields.load()  # load the dataset into memory

# Convert to SGRID-compliant dataset and create FieldSet
fields = {"U": ds_fields["uo"], "V": ds_fields["vo"]}
ds_fset = parcels.convert.copernicusmarine_to_sgrid(fields=fields)
fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)

## Defining initial `time` as particle release


Defining the initial times of particles is done when the `ParticleSet` is defined. Although `time` and `z` are optional arguments (with FieldSet t=0 and z=0 as defaults), it is good practice to define them explicitly to ensure expected behavior. The simplest way to delay the start of a particle is to use the `time` argument for each particle.


In [ ]:
npart = 10  # number of particles to be released
lon = 32 * np.ones(npart)
lat = np.linspace(-31.5, -30.5, npart, dtype=np.float32)
# release every particle one hour later from the initial fieldset time
time = ds_fields.time.values[0] + np.arange(0, npart) * np.timedelta64(1, "h")
z = np.repeat(ds_fields.depth.values[0], npart)

pset = parcels.ParticleSet(
    fieldset=fieldset, pclass=parcels.Particle, lon=lon, lat=lat, time=time, z=z
)

Then we can execute the `pset` as usual


In [ ]:
output_file = parcels.ParticleFile(
    "delayparticle_time.parquet", outputdt=np.timedelta64(1, "h")
)

pset.execute(
    parcels.kernels.AdvectionRK2,
    runtime=np.timedelta64(24, "h"),
    dt=np.timedelta64(5, "m"),
    output_file=output_file,
)

And then finally, we can show a movie of the particles. Note that the southern-most particles start to move first.


In [ ]:
df_particles = parcels.read_particlefile("delayparticle_time.parquet")

fig = plt.figure(figsize=(7, 5), constrained_layout=True)
ax = fig.add_subplot()

ax.set_ylabel("Latitude [deg N]")
ax.set_xlabel("Longitude [deg E]")
ax.set_xlim(31, 33)
ax.set_ylim(-32, -30)

timerange = df_particles["time"].unique()

particles = df_particles.filter(pl.col("time") == pl.lit(timerange[0]))
sc = ax.scatter(particles["lon"], particles["lat"])
title = ax.set_title(f"Particles at t = {timerange[0]}")


def animate(i):
    particles = df_particles.filter(pl.col("time") == pl.lit(timerange[i]))
    sc.set_offsets(np.c_[particles["lon"], particles["lat"]])
    title.set_text(f"Particles at t = {timerange[i]}")


anim = FuncAnimation(fig, animate, frames=len(timerange), interval=100)
plt.close()
anim

## Release particles repeatedly at a set frequency using `np.broadcast_to`


When you want to repeatedly release particles from the same set of locations, you can expand the initial array of particle release locations. Here we show how to release particles at a set frequency `repeatdt`, for a fixed number of releases `nrepeat`. The total number of particles released is then **nrepeat** x **npart**.

<div class="alert alert-info">

Note that the `repeatdt` argument in `parcels.ParticleSet` used in previous versions of parcels is no longer supported as of v4
</div>

In [ ]:
npart = 10  # number of particles to be released

lon_i = 32 * np.ones(npart)
lat_i = np.linspace(-31.5, -30.5, npart, dtype=np.float32)
time_i = np.repeat(ds_fields.time.values[0], npart)
z_i = np.repeat(ds_fields.depth.values[0], npart)

# repeat release at frequency `repeatdt` for `nrepeat` different releases
repeatdt = np.timedelta64(6, "h")
nrepeat = 3

lon = np.broadcast_to(lon_i, (nrepeat, npart))
lat = np.broadcast_to(lat_i, (nrepeat, npart))
time = (
    np.broadcast_to(time_i, (nrepeat, npart))
    + np.arange(0, nrepeat)[:, np.newaxis] * repeatdt
)
print(f"pset.time shape = {time.shape}")
z = np.broadcast_to(z_i, (nrepeat, npart))

pset = parcels.ParticleSet(
    fieldset=fieldset, pclass=parcels.Particle, lon=lon, lat=lat, time=time, z=z
)

Now we again define an output file and execute the `pset` as usual.


In [ ]:
output_file = parcels.ParticleFile(
    "delayparticle_releasedt.parquet", outputdt=np.timedelta64(1, "h")
)

pset.execute(
    parcels.kernels.AdvectionRK2,
    runtime=np.timedelta64(24, "h"),
    dt=np.timedelta64(5, "h"),
    output_file=output_file,
)

And we get an animation where a new particle is released every 6 hours from each start location


In [ ]:
df_particles = parcels.read_particlefile("delayparticle_releasedt.parquet")

fig = plt.figure(figsize=(7, 5), constrained_layout=True)
ax = fig.add_subplot()

ax.set_ylabel("Latitude [deg N]")
ax.set_xlabel("Longitude [deg E]")
ax.set_xlim(31, 33)
ax.set_ylim(-32, -30)

timerange = df_particles["time"].unique()

particles = df_particles.filter(pl.col("time") == pl.lit(timerange[0]))
sc = ax.scatter(particles["lon"], particles["lat"])
title = ax.set_title(f"Particles at t = {timerange[0]}")


def animate(i):
    particles = df_particles.filter(pl.col("time") == pl.lit(timerange[i]))
    sc.set_offsets(np.c_[particles["lon"], particles["lat"]])
    title.set_text(f"Particles at t = {timerange[i]}")


anim = FuncAnimation(fig, animate, frames=len(timerange), interval=100)
plt.close(fig)
anim

## Synced `time` in the output file

Note that, because the `outputdt` variable controls the Kernel-loop, all particles are written _at the same time_, even when they start at a non-multiple of `outputdt`.

For example, if your particles start at `time=[0, 1, 2]` and `outputdt=2`, then the times written (for `dt=1` and `endtime=4`) will be `[0, 2, 2, 2, 4, 4, 4]`

In [ ]:
outfilepath = "delayparticle_nonmatchingtime.parquet"

pset = parcels.ParticleSet(
    fieldset=fieldset,
    pclass=parcels.Particle,
    lat=[-31] * 3,
    lon=[32] * 3,
    time=ds_fields.time.values[0] + np.arange(3) * np.timedelta64(1, "h"),
    z=[0.5] * 3,
)

output_file = parcels.ParticleFile(outfilepath, outputdt=np.timedelta64(2, "h"))
pset.execute(
    parcels.kernels.AdvectionRK2,
    endtime=ds_fields.time.values[0] + np.timedelta64(4, "h"),
    dt=np.timedelta64(1, "h"),
    output_file=output_file,
)

And indeed, the `time` values in the output file are as expected


In [ ]:
outtime_infile = parcels.read_particlefile(outfilepath)
print(outtime_infile["time"])

Now, for some applications, this behavior may be undesirable; for example when particles need to be analyzed at a same age (instead of at a same time). In that case, we recommend either changing `outputdt` so that it is a common divisor of all start times; or doing multiple Parcels runs with subsets of the original `ParticleSet` (e.g., in the example above, one run with the Particles that start at `time=[0, 2]` and one with the Particle at `time=[1]`). In that case, you will get two files:


In [ ]:
for times in np.array([[0, 2], [1, 3]]).astype("timedelta64[h]"):
    pset = parcels.ParticleSet(
        fieldset=fieldset,
        pclass=parcels.Particle,
        lat=[-31] * len(times),
        lon=[32] * len(times),
        time=ds_fields.time.values[0] + times,
        z=[0.5] * len(times),
    )
    outfilepath = f"delayparticle_nonmatchingtime_{times[0]}_{times[1]}.parquet"
    output_file = parcels.ParticleFile(outfilepath, outputdt=np.timedelta64(2, "h"))
    pset.execute(
        parcels.kernels.AdvectionRK2,
        runtime=np.timedelta64(4, "h"),
        dt=np.timedelta64(1, "h"),
        output_file=output_file,
        verbose_progress=False,
    )
    outtime_infile = parcels.read_particlefile(outfilepath)
    print(outtime_infile["time"])